# 12. Verification Completion & FE-v2 Pipeline (`<= 120` Features) on Google Colab GPU

Notebook tự động giải nén `mbd_colab_bundle.zip` + `mbd_colab_patch_core.zip` từ Google Drive và chạy toàn bộ các bước còn lại trên Colab GPU/High-RAM:
1. **Group 2.4 (`FE-v1` OOF Permutation Importance):** Chạy trên GPU chỉ mất ~2–4 phút.
2. **Group 3 (`Multi-Seed`):** Hoàn tất 3 seeds cho `L5c_cb`, 5 seeds cho `L5c_lgbm_dialog_meta`, và 5 seeds cho `B_catboost_TGD_tuned` (Protocol B).
3. **FE-v2 (`<= 120` features max):**
   - Stage A: Chọn lọc & khử đa cộng tuyến (`|r| > 0.985`) **chỉ trên Validation**, khóa đúng `<= 120` biến.
   - Stage B: Kiểm chứng & tuning đa họ mô hình (`LogReg`, `LightGBM`, `XGBoost`, `CatBoostTuned`, kèm `scale_pos_weight`) + Blend trên Validation (`Protocol B`, 3 seeds).
   - Stage C: Xác nhận trên **Locked Test (`2022-11..2023-01`)** qua **5 seeds (`42, 142, 242, 342, 442`)** trên cả `Protocol B` & `Protocol A`, kèm Bootstrap ghép cặp (Holm & BH), Calibration Slope/Intercept, và Shuffled-Label Sanity.
4. Đóng gói toàn bộ output nhẹ (`outputs/fe_v2/` + `outputs/verification_audit/*.csv`) về lại Google Drive (`mbd_colab_results.zip`) để tải xuống viết báo cáo & xuất slide.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

!nvidia-smi || true
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "catboost", "lightgbm", "xgboost", "scikit-learn", "pandas", "pyarrow", "scipy", "duckdb", "matplotlib"
])

In [ ]:
# Giải nén bundle + patch từ Google Drive vào /content/mbd_work (chạy cực nhanh trên SSD local của Colab)
DRIVE_DIR = Path("/content/drive/MyDrive")
BUNDLE_ZIP = DRIVE_DIR / "mbd_colab_bundle.zip"
PATCH_ZIP = DRIVE_DIR / "mbd_colab_patch_core.zip"
LAG_CACHE_ZIP = DRIVE_DIR / "mbd_colab_group1_lag_cache.zip"  # Tùy chọn nếu muốn chạy lại CatBoost lag 7/14/30
WORK_DIR = Path("/content/mbd_work")

assert BUNDLE_ZIP.exists(), f"Không thấy {BUNDLE_ZIP}"
assert PATCH_ZIP.exists(), f"Không thấy {PATCH_ZIP}"

if not (WORK_DIR / "outputs" / "fe_v2" / "panel_fe_v2_candidates.parquet").exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    print("Extracting mbd_colab_bundle.zip ...")
    !unzip -q -o "{BUNDLE_ZIP}" -d "{WORK_DIR}"
    print("Extracting mbd_colab_patch_core.zip ...")
    !unzip -q -o "{PATCH_ZIP}" -d "{WORK_DIR}"
    if LAG_CACHE_ZIP.exists():
        print("Extracting optional mbd_colab_group1_lag_cache.zip ...")
        !unzip -q -o "{LAG_CACHE_ZIP}" -d "{WORK_DIR}"
else:
    print("Workspace already extracted at", WORK_DIR)

%cd /content/mbd_work
!ls -lh outputs/fe_v2/panel_fe_v2_candidates.parquet outputs/baseline_moscow_v1/dialog_emb_moscow.npz

In [ ]:
# 1. Chạy Group 2.4: FE-v1 139-feature OOF Permutation Importance (Val & Test)
!python -B -X utf8 src/scripts/verify_group2_utc_and_pi.py

In [ ]:
# 2. Chạy Group 3: Hoàn tất đa seed cho L5c_cb (3 seeds), L5c_lgbm (5 seeds), Protocol B B_catboost_TGD_tuned (5 seeds)
!python -B -X utf8 src/scripts/verify_group3_multiseed.py

In [ ]:
# 3. Chạy toàn bộ FE-v2 Pipeline (Stage A Val feature selection <= 120 + Stage B Val verify/tuning + Stage C 5-seed Locked Test)
!python -B -X utf8 src/scripts/run_fe_v2_pipeline.py --stage all

In [ ]:
# 4. (Tùy chọn) Nếu đã upload mbd_colab_group1_lag_cache.zip, chạy thêm CatBoost cho Group 1 lag 7/14/30d & T+2
import os
if os.path.exists("outputs/verification_audit/group1_pca_cache/lag7/_READY.flag"):
    print("Optional: running CatBoost Group 1 checks ...")
    !python -B -X utf8 src/scripts/verify_group1_models_fast.py --families cb
else:
    print("Skipping optional CatBoost Group 1 lag cache (LightGBM full sweep + Val CB ablations already present).")

In [ ]:
# 5. Đóng gói kết quả gọn nhẹ lên Google Drive để tải về máy viết báo cáo & slide
OUT_ZIP = "/content/drive/MyDrive/mbd_colab_results.zip"
!zip -q -r "{OUT_ZIP}" \
    outputs/verification_audit/*.csv \
    outputs/verification_audit/*.json \
    outputs/fe_v2/*.csv \
    outputs/fe_v2/*.json \
    outputs/fe_v2/predictions
!ls -lh "{OUT_ZIP}"